In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import layers

In [3]:
(X_train, _), (_, _) = keras.datasets.fashion_mnist.load_data()

X_train = X_train.astype('float32')/255
X_train = X_train.reshape(-1,28,28,1)

In [5]:
#build generator 
latend_dim = 100

generator = keras.Sequential([
    layers.Input(shape=(latend_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(28*28, activation='sigmoid'),
    layers.Reshape((28,28,1))
])
generator.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 128)               12928     
                                                                 
 dense_3 (Dense)             (None, 784)               101136    
                                                                 
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
Total params: 114,064
Trainable params: 114,064
Non-trainable params: 0
_________________________________________________________________


In [8]:
#build discriminator
discriminator = keras.Sequential([
    layers.Input(shape=(28,28,1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

discriminator.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense_6 (Dense)             (None, 128)               100480    
                                                                 
 dense_7 (Dense)             (None, 1)                 129       
                                                                 
Total params: 100,609
Trainable params: 100,609
Non-trainable params: 0
_________________________________________________________________


In [ ]:

discriminator.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

In [10]:
discriminator.trainable = False 

gan_input = keras.Input(shape=(latend_dim,))

fake_images = generator(gan_input)

gan_output = discriminator(fake_images)

gan = keras.Model(gan_input, gan_output)

gan.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)


In [13]:
epochs = 5000
batch_size = 128

for epoch in range(epochs):
    #train discriminator
    idx = np.random.randint(0,X_train.shape[0],batch_size)
    real_images = X_train[idx]

    noise = np.random.normal(0,1,(batch_size,latend_dim))
    fake_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size,1)))
    d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size,1)))

    #train generator
    noise = np.random.normal(0,1,(batch_size,latend_dim))
    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if epoch % 1000 == 0:
        print(f"Epoch: {epoch} |"
              f"D Loss: {d_loss_real[0]:.4f} |"
              f"G Loss: {g_loss[0]:.4f}"
        )


Epoch: 0 |D Loss: 0.4886 |G Loss: 0.9372
Epoch: 1000 |D Loss: 0.1271 |G Loss: 2.3325
Epoch: 2000 |D Loss: 0.1204 |G Loss: 2.6940
Epoch: 3000 |D Loss: 0.2880 |G Loss: 1.7775
Epoch: 4000 |D Loss: 0.1905 |G Loss: 2.2355


In [14]:
noise = np.random.normal(0,1,(10,latend_dim))
generated_images = generator.predict(noise,verbose=0)